# Khmer speech-to-text: fine-tuning Whisper-small (final project)

Fine-tunes `openai/whisper-small` (241M trainable parameters) on the Khmer ASR mini dataset
(OpenSLR 42, km-speech-corpus, kheng.info, MPWT, and our own RFI clips), then compares it with
the zero-shot baseline and an ablation model on two test sets.

**Setup (once)**
1. On your laptop: `python kaggle/package.py`, then upload as two **private** Kaggle datasets:
   `outputs/kaggle/khmer-asr-mini-data.zip` → `khmer-asr-mini-data`, and
   `outputs/kaggle/khmer-asr-mini-code.zip` → `khmer-asr-mini-code`.
2. Here: **Add Input** both datasets. Settings: **Accelerator → GPU T4 ×2**, **Internet → On**.

**Runs** — set `MODE`, then **Save Version → Save & Run All (Commit)**:

| Order | `MODE` | What it does |
| --- | --- | --- |
| 1 | `smoke` | 60 steps + 1 evaluation (~10 min): checks the setup, prints the speed |
| 2 | `train` | Main model on all sources (~17 h of audio, 5 epochs) |
| 3 | `ablation` | Same recipe on OpenSLR 42 only (~4 h, one speaker) |
| 4 | `evaluate` | Baseline, main and ablation models on `test` and `ddd_test` |

Before runs 3 and 4, **Add Input → Your Work** → the output of the earlier versions, so the
trained models are available. If a `train` run stops at the time limit, attach its output and
commit `train` again: it resumes from the last checkpoint.

In [ ]:
# ==== Settings ====
MODE = "smoke"          # "smoke" | "train" | "ablation" | "evaluate"
USE_BOTH_GPUS = True    # T4 x2 via torchrun; set False if the smoke run fails with a distributed error
TIME_LIMIT_HOURS = 11   # checkpoint and stop before Kaggle's 12 h limit
EPOCHS = 5
EVAL_LIMIT = 0          # evaluate mode: 0 = all clips; e.g. 500 to save GPU quota

In [ ]:
import glob, json, os, shutil, subprocess
from pathlib import Path

code_hits = glob.glob("/kaggle/input/**/src/train.py", recursive=True)
data_hits = glob.glob("/kaggle/input/**/khmer-asr-mini/summary.json", recursive=True)
assert code_hits, "Attach the khmer-asr-mini-code dataset (Add Input)"
assert data_hits, "Attach the khmer-asr-mini-data dataset (Add Input)"
PROJECT = Path("/tmp/project")  # code outside /kaggle/working so it isn't saved as output
shutil.copytree(Path(code_hits[0]).parents[1], PROJECT, dirs_exist_ok=True)
os.chdir(PROJECT)
DATA_DIR = Path(data_hits[0]).parent
WORK = Path("/kaggle/working")
print("Code:", Path(code_hits[0]).parents[1], "| Data:", DATA_DIR)
print(json.dumps(json.loads(Path(data_hits[0]).read_text())["splits"], indent=1)[:1500])

In [ ]:
!pip install -q -r requirements-train.txt
import torch
GPUS = torch.cuda.device_count()
assert GPUS, "No GPU: set Accelerator to GPU T4 x2"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(GPUS)])
assert tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 6), "torch < 2.6: restart kernel"

In [ ]:
RUNS = {  # mode -> (training sources, checkpoint dir, final model dir)
    "train": ("", WORK / "checkpoints-main", WORK / "models" / "whisper-small-khmer-mini"),
    "ablation": ("openslr42", WORK / "checkpoints-ablation", WORK / "models" / "whisper-small-khmer-openslr-only"),
}

def step_of(path):
    return int(Path(path).name.split("-")[-1])

# Bring back checkpoints from an earlier version's output, so a stopped run resumes.
if MODE in RUNS:
    checkpoint_dir = RUNS[MODE][1]
    previous = sorted(glob.glob(f"/kaggle/input/**/{checkpoint_dir.name}/checkpoint-*", recursive=True), key=step_of)
    if previous and not checkpoint_dir.exists():
        shutil.copytree(Path(previous[-1]).parent, checkpoint_dir)
        print("Resuming from", previous[-1])

two_gpus = USE_BOTH_GPUS and GPUS > 1
launcher = f"torchrun --nproc_per_node {GPUS} -m" if two_gpus else "python -m"
# Effective batch 32 either way: 16 per GPU x 2 GPUs, or 16 x 2 accumulation steps on one GPU.
common = f"--data-dir {DATA_DIR} --batch-size 16 --gradient-checkpointing --num-workers 4" + ("" if two_gpus else " --grad-accum 2")

if MODE == "smoke":
    command = (f"{launcher} src.train {common} --epochs 0.1 --eval-steps 30 --eval-samples 64 "
               f"--output-dir {WORK}/smoke --final-dir {WORK}/smoke-model")
elif MODE in RUNS:
    sources, checkpoint_dir, final_dir = RUNS[MODE]
    command = (f"{launcher} src.train {common} --epochs {EPOCHS} --eval-steps 500 "
               f"--time-limit-hours {TIME_LIMIT_HOURS} --output-dir {checkpoint_dir} --final-dir {final_dir} --resume"
               + (f" --sources {sources}" if sources else ""))
if MODE != "evaluate":
    print(command)
    !{command}

In [ ]:
if MODE == "evaluate":
    def find_model(name):
        hits = glob.glob(f"/kaggle/input/**/models/{name}/config.json", recursive=True) + glob.glob(f"{WORK}/models/{name}/config.json")
        return str(Path(hits[0]).parent) if hits else None

    models = {"baseline (zero-shot)": "openai/whisper-small",
              "main (all sources)": find_model("whisper-small-khmer-mini"),
              "ablation (OpenSLR only)": find_model("whisper-small-khmer-openslr-only")}
    for label, model in models.items():
        if model is None:
            print("Skipping", label, "- attach the output of the version that trained it")
            continue
        for split in ("test", "ddd_test"):
            limit = f"--limit {EVAL_LIMIT}" if EVAL_LIMIT else ""
            report = f"{label.split()[0]}_{split}.json"
            !python -m src.evaluate --model {model} --split {split} --data-dir {DATA_DIR} --batch-size 32 {limit} --output {report}
    shutil.copytree(PROJECT / "outputs" / "evaluation", WORK / "evaluation", dirs_exist_ok=True)

In [ ]:
# Session summary
if MODE == "smoke":
    states = sorted(glob.glob(f"{WORK}/smoke/checkpoint-*/trainer_state.json"), key=lambda p: step_of(Path(p).parent))
    if states:
        for entry in json.loads(Path(states[-1]).read_text())["log_history"]:
            if "eval_cer_no_space" in entry or "train_runtime" in entry:
                print(entry)
    print("Speed check: train_runtime / steps = seconds per step; x ~3,000 steps = a full 5-epoch run.")
    shutil.rmtree(WORK / "smoke", ignore_errors=True)
    shutil.rmtree(WORK / "smoke-model", ignore_errors=True)
elif MODE in RUNS:
    final_dir = RUNS[MODE][2]
    if (final_dir / "config.json").exists():
        print("Finished:", final_dir)
        shutil.rmtree(RUNS[MODE][1], ignore_errors=True)  # checkpoints no longer needed; keeps the output small
    else:
        print("Stopped at the time limit: attach this version's output and commit the same MODE again.")
elif MODE == "evaluate":
    rows = []
    for report in sorted((WORK / "evaluation").glob("*.json")):
        data = json.loads(report.read_text())
        rows.append((report.stem, data["clips"], *(round(data["metrics"][k], 4) for k in ("wer", "cer", "cer_no_space"))))
    print(f"{'run':32s} {'clips':>6s} {'WER':>8s} {'CER':>8s} {'CER no space':>13s}")
    for row in rows:
        print(f"{row[0]:32s} {row[1]:6d} {row[2]:8.4f} {row[3]:8.4f} {row[4]:13.4f}")
print(subprocess.run(["du", "-sh", str(WORK)], capture_output=True, text=True).stdout)